# AstroAI: Interactive Model Training Pipeline

This notebook contains the complete pipeline to load Kepler light curve data, preprocess it, build the custom `DeboDetectorNet` model (1D CNN + Bidirectional GRU), train it, and visually analyze the training metrics, confusion matrix, and classification report.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Add parent directory to path to resolve debo_model imports
sys.path.append(os.path.dirname(os.path.abspath('')))

from debo_model.dataset import prepare_splits
from debo_model.model import DeboDetectorNet

## 1. Load and Split preprocessed Kepler Data

We load the downloaded `.npz` files using the labels defined in `dataset_index.csv`. They are preprocessed, aligned, and split into train, validation, and test subsets.

In [ ]:
print("Loading and preprocessing dataset splits...")
(x_train, y_train), (x_val, y_val), (x_test, y_test) = prepare_splits(
    index_path="debo_model/dataset_index.csv" if os.path.exists("debo_model/dataset_index.csv") else "dataset_index.csv",
    target_len=2000,
    test_size=0.15,
    val_size=0.15,
    seed=42
)

print("\nDataset splits loaded successfully:")
print(f" - Train set: X={x_train.shape}, y={y_train.shape}")
print(f" - Val set:   X={x_val.shape}, y={y_val.shape}")
print(f" - Test set:  X={x_test.shape}, y={y_test.shape}")

## 2. Initialize DeboDetectorNet Model

Let's build the model and print the layer summary.

In [ ]:
model = DeboDetectorNet(input_len=2000, dropout=0.3)

# Trigger a forward call on dummy data to construct internal layer shapes
dummy_input = tf.zeros((1, 2000, 1))
_ = model(dummy_input)
model.summary()

## 3. Train the Model

We compile the model using the Adam optimizer and Binary Crossentropy, and add callbacks for early stopping, learning rate plateau schedules, and best model weight saving.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

checkpoint_dir = "saved_models"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, "debo_detector_best_nb.weights.h5")

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
)
]

epochs = 15
batch_size = 16

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1
)

## 4. Plot Training History (Loss & Accuracy)

Let's visualize the training curves to check if the model is learning successfully and to check for overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Plot Loss curves
axes[0].plot(history.history['loss'], label='Train Loss', color='crimson', lw=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='orange', lw=2, linestyle='--')
axes[0].set_title('Training & Validation Loss', fontsize=14)
axes[0].set_xlabel('Epochs', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. Plot Accuracy curves
axes[1].plot(history.history['accuracy'], label='Train Accuracy', color='dodgerblue', lw=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', color='green', lw=2, linestyle='--')
axes[1].set_title('Training & Validation Accuracy', fontsize=14)
axes[1].set_xlabel('Epochs', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 5. Evaluate on Test set & Generate Classification Report

Now we evaluate the model on unseen test data, and display the F1-Score, Recall, Precision, Confusion Matrix, and the full classification report.

In [ ]:
# Load best weights saved during training
if os.path.exists(checkpoint_path):
    model.load_weights(checkpoint_path)
    print("Loaded best weights from checkpoint.")

# Standard Keras evaluation
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

# Make predictions
y_pred_prob = model.predict(x_test, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)
y_true = y_test.astype(int)

# Calculate classification metrics
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("=" * 50)
print(f"Test Loss:            {test_loss:.4f}")
print(f"Test Accuracy:        {test_acc*100:.2f}%")
print(f"Precision:            {precision*100:.2f}%")
print(f"Recall (Sensitivity): {recall*100:.2f}%")
print(f"F1-Score:             {f1*100:.2f}%")
print("=" * 50)

# Print Confusion Matrix Table
print("\nConfusion Matrix / Classification Table:")
print(f"{'':<20} | {'Predicted Non-Planet (0)':<25} | {'Predicted Planet (1)':<25}")
print("-" * 78)
print(f"{'Actual Non-Planet (0)':<20} | {f'TN: {tn}':<25} | {f'FP: {fp}':<25}")
print(f"{'Actual Planet (1)':<20} | {f'FN: {fn}':<25} | {f'TP: {tp}':<25}")
print("-" * 78)

# Print detailed classification report
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=["Non-Planet", "Planet"], zero_division=0))